# 06 Sequence Models: RNN, LSTM & PyTorch Neural Architectures

**Scenario**: PyTorch Sentiment Classifier & Step-by-Step Scalar LSTM Forward Pass.

This notebook demonstrates step-by-step scalar LSTM forward pass computation and building a PyTorch bidirectional LSTM module.

## Step 1: Step-by-Step Scalar LSTM Forward Pass Computation

In [1]:
import numpy as np

# Scalar Inputs
x_t = 1.0
h_prev = 0.5
c_prev = 0.8

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Input activation sum z
z = h_prev + x_t  # 1.5

# Gating Signals
f_t = sigmoid(z)        # 0.8176
i_t = sigmoid(z)        # 0.8176
c_tilde = np.tanh(z)    # 0.9051

# Cell State Update
c_t = (f_t * c_prev) + (i_t * c_tilde)

# Output Gate & Hidden State Output
o_t = sigmoid(z)
h_t = o_t * np.tanh(c_t)

print("=== Scalar LSTM Forward Pass Results ===")
print(f"Forget Gate f_t: {f_t:.4f}")
print(f"Input Gate i_t : {i_t:.4f}")
print(f"Candidate C~_t : {c_tilde:.4f}")
print(f"Updated Cell State C_t : {c_t:.4f}")
print(f"Updated Hidden State h_t: {h_t:.4f}")

=== Scalar LSTM Forward Pass Results ===
Forget Gate f_t: 0.8176
Input Gate i_t : 0.8176
Candidate C~_t : 0.9051
Updated Cell State C_t : 1.3941
Updated Hidden State h_t: 0.7228


## Step 2: PyTorch Bidirectional LSTM Pipeline

In [2]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size=1000, embed_dim=32, hidden_dim=64, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hn, cn) = self.lstm(embedded)
        final_hidden = torch.cat((hn[-2], hn[-1]), dim=1)
        return self.fc(final_hidden)

model = SentimentLSTM()
dummy_batch = torch.randint(0, 1000, (4, 16))  # Batch of 4 sequences, length 16
logits = model(dummy_batch)

print("=== PyTorch SentimentLSTM Execution ===")
print("Input Batch Shape :", dummy_batch.shape)
print("Logits Tensor Shape:", logits.shape)
print("Logits Matrix:\n", logits.detach().numpy().round(4))

=== PyTorch SentimentLSTM Execution ===
Input Batch Shape : torch.Size([4, 16])
Logits Tensor Shape: torch.Size([4, 2])
Logits Matrix:
 [[ 0.0732  0.1702]
 [ 0.0225  0.0356]
 [-0.0048 -0.1164]
 [ 0.098   0.1305]]


## Output Explanation & Complexity Analysis

- **Scalar LSTM Calculation**: Inputs ($x_t=1.0, h_{t-1}=0.5$) produce updated cell state $C_t = 1.3941$ and hidden state $h_t = 0.7228$.
- **PyTorch BiLSTM Execution**: Concatenates forward and backward hidden states to output a 2-class logits matrix for batch size 4.
- **Time Complexity**: $O(T \cdot d^2)$ sequential steps.
- **Space Complexity**: $O(4 \cdot d^2)$ parameter weights.